# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Notebook 8 – Preparing the Quran Passage Collection for Retrieval (DD-009)

## Introduction

The previous notebooks focused on constructing a linguistically informed query representation through Arabic query preprocessing, hybrid root extraction, and semantic enrichment using *Maqāyīs al-Lughah*. The resulting semantically enriched queries constitute one side of the proposed retrieval framework.

The present notebook shifts the focus to the document collection by preparing the Quran passages that will be searched during retrieval. The QuranQA Task A benchmark provides the **Quran Passage Collection (QPC)**, which contains the passages used as the retrieval corpus for evaluating question answering systems.

Rather than modifying the original Quranic text, this notebook explores the structure of the passage collection, validates its contents, analyzes its characteristics, and prepares an efficient passage representation for the subsequent retrieval stage. Preserving the official benchmark collection ensures reproducibility while enabling efficient access to the passages during retrieval.

---

## Objectives

The objectives of this notebook are to:

1. Load the QuranQA Task A Quran Passage Collection (QPC).
2. Explore the structure and attributes of the passage dataset.
3. Examine representative Quran passages.
4. Analyze descriptive statistics of the passage collection.
5. Construct an efficient passage lookup structure for retrieval.
6. Export the processed passage collection for the retrieval stage.

---

## Expected Deliverables

At the completion of this notebook, the following deliverables are produced:

- A comprehensive understanding of the QuranQA passage collection.
- Descriptive statistics of the Quran passages.
- A structured passage lookup representation.
- A processed Quran passage collection prepared for retrieval.
- A documented document-side foundation for the proposed retrieval framework.

---

## Design Decision (DD-009)

The proposed framework preserves the official QuranQA passage collection without modifying the Quranic text. Instead, the passages are organized into an efficient lookup structure that supports fast and reproducible retrieval while maintaining full compatibility with the benchmark dataset.

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from IPython.display import display

# ------------------------------------------------------------
# Display Options
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("=" * 70)
print("LIBRARIES LOADED")
print("=" * 70)

print("✓ pathlib")
print("✓ json")
print("✓ numpy")
print("✓ pandas")
print("✓ IPython.display")

LIBRARIES LOADED
✓ pathlib
✓ json
✓ numpy
✓ pandas
✓ IPython.display


In [2]:
# ============================================================
# Define Project Paths
# ============================================================

# Resolve the current Writing Workspace without a personal absolute path.
NOTEBOOK_DIR = Path.cwd().resolve()


def find_workspace_root(start_path):
    """Locate the thesis Writing Workspace from the execution directory."""
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        authority_file = (
            candidate /
            "00_Project_Control" /
            "README_Workspace_Authority.txt"
        )
        qpc_file = (
            candidate /
            "02_Methodology_Evidence" /
            "Data and Resources" /
            "QuranQA" /
            "QQA23_TaskA_QPC_v1.1.tsv"
        )

        if authority_file.exists() and qpc_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the authoritative thesis Writing Workspace "
        "from the current execution directory."
    )


PROJECT_ROOT = find_workspace_root(NOTEBOOK_DIR)

# ------------------------------------------------------------
# Current Methodology-Evidence Directories
# ------------------------------------------------------------

DATA_RESOURCES_DIR = (
    PROJECT_ROOT /
    "02_Methodology_Evidence" /
    "Data and Resources"
)

QURANQA_DIR = DATA_RESOURCES_DIR / "QuranQA"

OUTPUT_DIR = (
    DATA_RESOURCES_DIR /
    "Corrected_Outputs" /
    "QPC_Passage_Side_Validation"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Input Files
# ------------------------------------------------------------

PASSAGE_COLLECTION_FILE = QURANQA_DIR / "QQA23_TaskA_QPC_v1.1.tsv"

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"Project Root              : {PROJECT_ROOT}")
print(f"QuranQA Directory         : {QURANQA_DIR}")
print(f"Output Directory          : {OUTPUT_DIR}")

print("\nInput Files")

print(f"Passage Collection        : {PASSAGE_COLLECTION_FILE.name}")

print("\nValidation")

print(f"Passage Collection Exists : {PASSAGE_COLLECTION_FILE.exists()}")

PROJECT PATHS
Project Root              : ${THESIS_WORKSPACE}
QuranQA Directory         : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Data and Resources\QuranQA
Output Directory          : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Data and Resources\Corrected_Outputs\QPC_Passage_Side_Validation

Input Files
Passage Collection        : QQA23_TaskA_QPC_v1.1.tsv

Validation
Passage Collection Exists : True


## Loading the Quran Passage Collection

The Quran Passage Collection (QPC) constitutes the document corpus used by the QuranQA Task A benchmark. Each record represents a passage from the Holy Quran together with its associated metadata.

This section loads the official passage collection into a pandas DataFrame and performs an initial inspection to verify that the dataset has been correctly imported. The structure of the dataset is examined before conducting more detailed exploratory analyses in the subsequent sections.

In [3]:
# ============================================================
# Load Quran Passage Collection
# ============================================================

# ------------------------------------------------------------
# Load Dataset (No Header in TSV)
# ------------------------------------------------------------

passages_df = pd.read_csv(
    PASSAGE_COLLECTION_FILE,
    sep="\t",
    header=None,
    names=["Passage_ID", "Passage_Text"]
)

# ------------------------------------------------------------
# Dataset Overview
# ------------------------------------------------------------

print("=" * 70)
print("QURAN PASSAGE COLLECTION LOADED")
print("=" * 70)

print(f"Total Passages : {len(passages_df):,}")
print(f"Total Columns  : {len(passages_df.columns)}")

print("\nColumns")

print(list(passages_df.columns))

print("\nSample Records")

display(passages_df.head())

QURAN PASSAGE COLLECTION LOADED
Total Passages : 1,266
Total Columns  : 2

Columns
['Passage_ID', 'Passage_Text']

Sample Records


,Passage_ID,Passage_Text
0,1:1-4,بسم الله الرحمن الرحيم. الحمد لله رب العالمين. الرحمن الرحيم. مالك يوم الدين.
1,1:5-6,إياك نعبد وإياك نستعين. اهدنا الصراط المستقيم.
2,1:7-7,صراط الذين أنعمت عليهم غير المغضوب عليهم ولا الضالين.
3,2:1-2,الم. ذلك الكتاب لا ريب فيه هدى للمتقين.
4,2:3-5,الذين يؤمنون بالغيب ويقيمون الصلاة ومما رزقناهم ينفقون. والذين يؤمنون بما أنزل إليك وما أنزل من قبلك وبالآخرة هم يوق...


## Exploring the Structure of the Quran Passage Collection

Before preparing the passage collection for retrieval, it is important to understand its structure and verify the integrity of the dataset. This section examines the data types, checks for missing or duplicate values, and validates the uniqueness of the passage identifiers.

Understanding these structural characteristics ensures that the passage collection is complete and suitable for use as the document corpus in the proposed retrieval framework.

In [4]:
# ============================================================
# Explore Quran Passage Collection Structure
# ============================================================

# ------------------------------------------------------------
# Dataset Information
# ------------------------------------------------------------

print("=" * 70)
print("PASSAGE COLLECTION STRUCTURE")
print("=" * 70)

print("\nDataset Shape")
print(f"Rows    : {passages_df.shape[0]:,}")
print(f"Columns : {passages_df.shape[1]}")

print("\nData Types")
display(passages_df.dtypes.to_frame(name="Data Type"))

# ------------------------------------------------------------
# Missing Values
# ------------------------------------------------------------

print("\nMissing Values")

missing_df = (
    passages_df
    .isnull()
    .sum()
    .rename("Missing Values")
    .to_frame()
)

display(missing_df)

# ------------------------------------------------------------
# Duplicate Passage IDs
# ------------------------------------------------------------

duplicate_ids = passages_df["Passage_ID"].duplicated().sum()

print(f"\nDuplicate Passage IDs : {duplicate_ids}")

# ------------------------------------------------------------
# Duplicate Passage Texts
# ------------------------------------------------------------

duplicate_texts = passages_df["Passage_Text"].duplicated().sum()

print(f"Duplicate Passages    : {duplicate_texts}")

# ------------------------------------------------------------
# Passage ID Validation
# ------------------------------------------------------------

unique_ids = passages_df["Passage_ID"].nunique()

print(f"Unique Passage IDs    : {unique_ids:,}")

print("\nSample Passage IDs")

display(passages_df["Passage_ID"].head(15))

PASSAGE COLLECTION STRUCTURE

Dataset Shape
Rows    : 1,266
Columns : 2

Data Types


,Data Type
Passage_ID,str
Passage_Text,str



Missing Values


,Missing Values
Passage_ID,0
Passage_Text,0



Duplicate Passage IDs : 0
Duplicate Passages    : 0
Unique Passage IDs    : 1,266

Sample Passage IDs


0       1:1-4
1       1:5-6
2       1:7-7
3       2:1-2
4       2:3-5
5       2:6-7
6      2:8-16
7     2:17-20
8     2:21-22
9     2:23-24
10    2:25-25
11    2:26-27
12    2:28-29
13    2:30-33
14    2:34-39
Name: Passage_ID, dtype: str

## Parsing Passage Identifiers

Each passage in the QuranQA Passage Collection is uniquely identified by a passage identifier that encodes the Surah number together with the range of Ayahs contained in the passage.

This section parses the passage identifiers into their individual components, including the Surah number, the starting Ayah, the ending Ayah, and the total number of Ayahs contained in each passage. These structural attributes facilitate later retrieval and analysis while preserving the original passage identifiers.

In [5]:
# ============================================================
# Parse Passage Identifiers
# ============================================================

# ------------------------------------------------------------
# Parse Passage_ID
# ------------------------------------------------------------

def parse_passage_id(pid):

    surah, ayah_range = pid.split(":")

    if "-" in ayah_range:

        start_ayah, end_ayah = ayah_range.split("-")

    else:

        start_ayah = end_ayah = ayah_range

    start_ayah = int(start_ayah)
    end_ayah = int(end_ayah)

    return pd.Series({

        "Surah": int(surah),
        "Start_Ayah": start_ayah,
        "End_Ayah": end_ayah,
        "Ayah_Count": end_ayah - start_ayah + 1

    })

# ------------------------------------------------------------
# Apply Parsing
# ------------------------------------------------------------

passages_df = pd.concat(

    [

        passages_df,

        passages_df["Passage_ID"].apply(parse_passage_id)

    ],

    axis=1

)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("=" * 70)
print("PASSAGE IDENTIFIERS PARSED")
print("=" * 70)

print(f"Total Passages : {len(passages_df):,}")

print(f"Unique Surahs  : {passages_df['Surah'].nunique()}")

print(f"Average Ayahs per Passage : {passages_df['Ayah_Count'].mean():.2f}")

print()

display(passages_df.head(10))

PASSAGE IDENTIFIERS PARSED
Total Passages : 1,266
Unique Surahs  : 114
Average Ayahs per Passage : 4.93



,Passage_ID,Passage_Text,Surah,Start_Ayah,End_Ayah,Ayah_Count
0,1:1-4,بسم الله الرحمن الرحيم. الحمد لله رب العالمين. الرحمن الرحيم. مالك يوم الدين.,1,1,4,4
1,1:5-6,إياك نعبد وإياك نستعين. اهدنا الصراط المستقيم.,1,5,6,2
2,1:7-7,صراط الذين أنعمت عليهم غير المغضوب عليهم ولا الضالين.,1,7,7,1
3,2:1-2,الم. ذلك الكتاب لا ريب فيه هدى للمتقين.,2,1,2,2
4,2:3-5,الذين يؤمنون بالغيب ويقيمون الصلاة ومما رزقناهم ينفقون. والذين يؤمنون بما أنزل إليك وما أنزل من قبلك وبالآخرة هم يوق...,2,3,5,3
5,2:6-7,إن الذين كفروا سواء عليهم أأنذرتهم أم لم تنذرهم لا يؤمنون. ختم الله على قلوبهم وعلى سمعهم وعلى أبصارهم غشاوة ولهم عذ...,2,6,7,2
6,2:8-16,ومن الناس من يقول آمنا بالله وباليوم الآخر وما هم بمؤمنين. يخادعون الله والذين آمنوا وما يخدعون إلا أنفسهم وما يشعرو...,2,8,16,9
7,2:17-20,مثلهم كمثل الذي استوقد نارا فلما أضاءت ما حوله ذهب الله بنورهم وتركهم في ظلمات لا يبصرون. صم بكم عمي فهم لا يرجعون. ...,2,17,20,4
8,2:21-22,يا أيها الناس اعبدوا ربكم الذي خلقكم والذين من قبلكم لعلكم تتقون. الذي جعل لكم الأرض فراشا والسماء بناء وأنزل من الس...,2,21,22,2
9,2:23-24,وإن كنتم في ريب مما نزلنا على عبدنا فأتوا بسورة من مثله وادعوا شهداءكم من دون الله إن كنتم صادقين. فإن لم تفعلوا ولن...,2,23,24,2


## Descriptive Analysis of the Passage Collection

To better understand the characteristics of the retrieval corpus, descriptive statistics are computed for the Quran passages. In addition to the structural metadata extracted from the passage identifiers, this analysis measures the textual length of each passage in both words and characters.

The resulting statistics provide an overview of the document collection and describe the distribution of passage lengths that will be processed during retrieval.

In [6]:
# ============================================================
# Descriptive Analysis of the Passage Collection
# ============================================================

# ------------------------------------------------------------
# Compute Passage Length Statistics
# ------------------------------------------------------------

passages_df["Character_Count"] = passages_df["Passage_Text"].str.len()

passages_df["Word_Count"] = passages_df["Passage_Text"].str.split().str.len()

# ------------------------------------------------------------
# Summary Statistics
# ------------------------------------------------------------

print("=" * 70)
print("PASSAGE COLLECTION STATISTICS")
print("=" * 70)

print(f"Average Characters per Passage : {passages_df['Character_Count'].mean():.2f}")
print(f"Minimum Characters             : {passages_df['Character_Count'].min()}")
print(f"Maximum Characters             : {passages_df['Character_Count'].max()}")

print()

print(f"Average Words per Passage      : {passages_df['Word_Count'].mean():.2f}")
print(f"Minimum Words                  : {passages_df['Word_Count'].min()}")
print(f"Maximum Words                  : {passages_df['Word_Count'].max()}")

# ------------------------------------------------------------
# Passages per Surah
# ------------------------------------------------------------

surah_summary = (
    passages_df
    .groupby("Surah")
    .size()
    .reset_index(name="Passage_Count")
)

print("\nSample Surah Statistics")

display(surah_summary.head(15))

print("\nSample Passage Statistics")

display(
    passages_df[
        [
            "Passage_ID",
            "Ayah_Count",
            "Word_Count",
            "Character_Count"
        ]
    ].head(20)
)

PASSAGE COLLECTION STATISTICS
Average Characters per Passage : 327.02
Minimum Characters             : 39
Maximum Characters             : 1086

Average Words per Passage      : 61.54
Minimum Words                  : 7
Maximum Words                  : 219

Sample Surah Statistics


,Surah,Passage_Count
0,1,3
1,2,103
2,3,50
3,4,66
4,5,43
5,6,45
6,7,54
7,8,18
8,9,39
9,10,28



Sample Passage Statistics


,Passage_ID,Ayah_Count,Word_Count,Character_Count
0,1:1-4,4,13,77
1,1:5-6,2,7,46
2,1:7-7,1,9,53
3,2:1-2,2,8,39
4,2:3-5,3,28,162
5,2:6-7,2,23,124
6,2:8-16,9,104,572
7,2:17-20,4,67,349
8,2:21-22,2,35,188
9,2:23-24,2,33,178


## Building the Passage Lookup Structure

To support efficient document retrieval, the processed Quran passages are organized into an in-memory lookup dictionary indexed by their unique passage identifiers.

Each dictionary entry stores the original passage text together with its structural metadata, including the Surah number, Ayah range, and passage statistics. This lookup structure provides constant-time access to any passage during retrieval while preserving the original QuranQA document collection without modification.

In [7]:
# ============================================================
# Build Passage Lookup Dictionary
# ============================================================

passage_lookup = {}

# ------------------------------------------------------------
# Build Dictionary
# ------------------------------------------------------------

for _, row in passages_df.iterrows():

    passage_lookup[row["Passage_ID"]] = {

        "Passage_Text": row["Passage_Text"],

        "Surah": row["Surah"],

        "Start_Ayah": row["Start_Ayah"],

        "End_Ayah": row["End_Ayah"],

        "Ayah_Count": row["Ayah_Count"],

        "Word_Count": row["Word_Count"],

        "Character_Count": row["Character_Count"]

    }

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("=" * 70)
print("PASSAGE LOOKUP DICTIONARY CREATED")
print("=" * 70)

print(f"Dictionary Entries : {len(passage_lookup):,}")

sample_keys = list(passage_lookup.keys())[:5]

print("\nSample Passage IDs")

for key in sample_keys:
    print(f"• {key}")

print("\nSample Lookup Entry")

first_key = sample_keys[0]

display(
    pd.DataFrame([passage_lookup[first_key]])
)

PASSAGE LOOKUP DICTIONARY CREATED
Dictionary Entries : 1,266

Sample Passage IDs
• 1:1-4
• 1:5-6
• 1:7-7
• 2:1-2
• 2:3-5

Sample Lookup Entry


,Passage_Text,Surah,Start_Ayah,End_Ayah,Ayah_Count,Word_Count,Character_Count
0,بسم الله الرحمن الرحيم. الحمد لله رب العالمين. الرحمن الرحيم. مالك يوم الدين.,1,1,4,4,13,77


## Exporting the Processed Passage Collection

The processed Quran passage collection is exported for use by the retrieval component of the proposed framework.

The exported dataset preserves the original passage identifiers and Quranic text while incorporating the structural metadata derived from the passage identifiers together with the computed passage statistics. This processed collection serves as the document corpus for the retrieval engine implemented in the subsequent notebook.

In [8]:
# ============================================================
# Export Processed Passage Collection
# ============================================================

OUTPUT_FILE = OUTPUT_DIR / "QuranQA_Processed_Passage_Collection.csv"

# ------------------------------------------------------------
# Export CSV
# ------------------------------------------------------------

passages_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Export Summary
# ------------------------------------------------------------

print("=" * 70)
print("PROCESSED PASSAGE COLLECTION EXPORTED")
print("=" * 70)

print(f"Passages Exported : {len(passages_df):,}")
print(f"Output File       : {OUTPUT_FILE.name}")
print(f"Location          : {OUTPUT_DIR}")

print("\nExport completed successfully.")

PROCESSED PASSAGE COLLECTION EXPORTED
Passages Exported : 1,266
Output File       : QuranQA_Processed_Passage_Collection.csv
Location          : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Data and Resources\Corrected_Outputs\QPC_Passage_Side_Validation

Export completed successfully.


## Notebook Summary

This notebook prepared the document collection used by the proposed retrieval framework by processing the official **QuranQA Task A Quran Passage Collection (QPC)**. While the previous notebooks focused on constructing a linguistically informed and semantically enriched query representation, this notebook established the document-side foundation required for passage retrieval.

The notebook began by loading the official Quran passage collection and validating its structural integrity. An exploratory analysis confirmed that the dataset contained **1,266** unique Quran passages covering all **114** Surahs, with no missing values or duplicate passage identifiers. This verification ensured that the benchmark collection could be used directly without additional cleaning or modification.

To enrich the structural representation of the passage collection, each passage identifier was parsed into its constituent metadata, including the Surah number, starting Ayah, ending Ayah, and the total number of Ayahs contained within each passage. Additional descriptive statistics were computed by measuring the textual length of each passage in both words and characters, providing a quantitative characterization of the retrieval corpus.

To support efficient retrieval, the processed passage collection was organized into an in-memory passage lookup dictionary indexed by the unique passage identifiers. Each dictionary entry stores the original Quranic passage together with its structural metadata and descriptive statistics, enabling constant-time access during retrieval while preserving the integrity of the official QuranQA benchmark.

Finally, the processed passage collection was exported as a structured dataset for use in the subsequent retrieval stage. Together with the semantically enriched query representation produced in Notebook 7, the processed passage collection constitutes the document component of the proposed retrieval framework and serves as the retrieval corpus for the implementation of the retrieval engine in the following notebook.